# JS02 Tugas Praktikum PBL - Klasifikasi Sleep Disorder

In [1]:
from pathlib import Path
import pickle
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv(Path('../Dataset/Sleep_health_and_lifestyle_dataset.csv'))
df['Sleep Disorder'] = df['Sleep Disorder'].fillna('None')
bp = df['Blood Pressure'].str.split('/', expand=True).astype(float)
df['Systolic BP'], df['Diastolic BP'] = bp[0], bp[1]
df = df.drop(columns=['Person ID', 'Blood Pressure'])
X, y = df.drop(columns='Sleep Disorder'), df['Sleep Disorder']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
numeric = X.select_dtypes(include=['number']).columns.tolist()
categorical = [col for col in X.columns if col not in numeric]
preprocessor = ColumnTransformer([('numeric', 'passthrough', numeric), ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical)])
model = Pipeline([('preprocessor', preprocessor), ('classifier', RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42))])
model.fit(X_train, y_train)
prediction = model.predict(X_test)
print(f'Akurasi: {accuracy_score(y_test, prediction):.2%}')
print(classification_report(y_test, prediction, zero_division=0))
print('Confusion matrix:\n', confusion_matrix(y_test, prediction, labels=['None', 'Insomnia', 'Sleep Apnea']))
with open('js02pbl_sleep_disorder_model.pkl', 'wb') as file: pickle.dump(model, file)

Akurasi: 86.67%
              precision    recall  f1-score   support

    Insomnia       0.70      0.93      0.80        15
        None       1.00      0.86      0.93        44
 Sleep Apnea       0.76      0.81      0.79        16

    accuracy                           0.87        75
   macro avg       0.82      0.87      0.84        75
weighted avg       0.89      0.87      0.87        75

Confusion matrix:
 [[38  3  3]
 [ 0 14  1]
 [ 0  3 13]]
